In [ ]:
import sys
sys.path.append("..")


In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("../grasp_analysis_results.csv")

# Function to calculate success rates
def calculate_success_rates(data):
    total = len(data)
    graspable = (data['is_graspable'] == True).sum() / total * 100
    collision_free = (data['has_collision'] == False).sum() / total * 100
    successful = ((data['is_graspable'] == True) & (data['has_collision'] == False)).sum() / total * 100
    
    return {
        'Total samples': total,
        'Graspable rate': f"{graspable:.2f}%",
        'Collision-free rate': f"{collision_free:.2f}%",
        'Overall success rate': f"{successful:.2f}%"
    }

# Split data into training and non-training sets
training_data = df[df['is_used_in_training'] == True]
non_training_data = df[df['is_used_in_training'] == False]

# Calculate and print results
print("Training Data Results:")
training_results = calculate_success_rates(training_data)
for metric, value in training_results.items():
    print(f"{metric}: {value}")

print("\nNon-Training Data Results:")
non_training_results = calculate_success_rates(non_training_data)
for metric, value in non_training_results.items():
    print(f"{metric}: {value}")

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("../grasp_analysis_results.csv")

# First let's look at the value counts of is_used_in_training
print("Distribution of is_used_in_training:")
print(df['is_used_in_training'].value_counts())
print("\nUnique values in is_used_in_training:")
print(df['is_used_in_training'].unique())

# Check the data types
print("\nData types of columns:")
print(df.dtypes)

# Look at a few rows
print("\nFirst few rows:")
print(df.head())

In [ ]:
training_data = df[df['is_used_in_training'] == True]
print(training_data.head())

In [ ]:
import json

with open("../logs/checkpoints/run_20250202_233846/used_grasp_files.json", "r") as f:
    used_files = json.load(f)

print(used_files[:10])

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

def extract_info_from_cache_files(used_files):
    """Extract info from all cache files at once."""
    cache_info = []
    for filename in used_files:
        parts = filename[:-3].split("_")  # Remove .h5
        cache_info.append({
            'item_name': "_".join(parts[:-2]),
            'item_id': parts[-2],
            'scale': float(parts[-1])
        })
    return pd.DataFrame(cache_info)

def extract_info_from_mesh_paths(df):
    """Extract item_name and item_id from all mesh paths at once."""
    paths = df['mesh_path'].apply(Path)
    return pd.DataFrame({
        'item_name': paths.apply(lambda p: p.parent.name),
        'item_id': paths.apply(lambda p: p.stem),
        'scale': df['dataset_mesh_scale']
    })

# Read the CSV
df = pd.read_csv("../grasp_analysis_results.csv")

# Load used_files (replace this with your loading method)
# used_files = ["item_name_item_id_scale.h5", ...]

# Extract info from both sources
cache_df = extract_info_from_cache_files(used_files)
mesh_df = extract_info_from_mesh_paths(df)

# Create boolean mask for matching entries
matches = pd.merge(
    mesh_df.assign(idx=np.arange(len(mesh_df))), 
    cache_df,
    on=['item_name', 'item_id'],
    suffixes=('_mesh', '_cache')
)
matches = matches[np.isclose(matches['scale_mesh'], matches['scale_cache'], rtol=1e-9)]

# Update is_used_in_training using the matches
df['is_used_in_training'] = False
df.loc[matches['idx'], 'is_used_in_training'] = True

# Save the updated CSV
df.to_csv("../grasp_analysis_results_updated.csv", index=False)

In [ ]:
import pandas as pd

def analyze_grasp_results(df, label=""):
    """Analyze grasp results for a given dataset."""
    total = len(df)
    if total == 0:
        print(f"\n{label} Results: No samples found")
        return
    
    metrics = {
        'Graspable rate': (df['is_graspable'] == True).mean() * 100,
        'Collision-free rate': (df['has_collision'] == False).mean() * 100,
        'Overall success rate': ((df['is_graspable'] == True) & 
                               (df['has_collision'] == False)).mean() * 100
    }
    
    print(f"\n{label} Results (Total samples: {total:,}):")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.2f}%")
    
    print("\nDetailed breakdown:")
    print(pd.crosstab(df['is_graspable'], df['has_collision']))

# Read the CSV
df = pd.read_csv("../grasp_analysis_results_updated.csv")

# Print overall statistics
total = len(df)
training = df['is_used_in_training'].sum()
print(f"Dataset Overview:")
print(f"Total samples: {total:,}")
print(f"Training samples: {training:,} ({training/total*100:.2f}%)")
print(f"Non-training samples: {total-training:,} ({(total-training)/total*100:.2f}%)")

# Analyze both sets
analyze_grasp_results(df[df['is_used_in_training']], "Training Data")
analyze_grasp_results(df[~df['is_used_in_training']], "Non-Training Data")

In [ ]:
import pandas as pd

def analyze_grasp_results(df, label=""):
    """Analyze grasp results for a given dataset."""
    total = len(df)
    if total == 0:
        print(f"\n{label} Results: No samples found")
        return
    
    metrics = {
        'Graspable rate': (df['is_graspable'] == True).mean() * 100,
        'Collision-free rate': (df['has_collision'] == False).mean() * 100,
        'Overall success rate': ((df['is_graspable'] == True) & 
                               (df['has_collision'] == False)).mean() * 100
    }
    
    print(f"\n{label} Results (Total samples: {total:,}):")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.2f}%")
    
    print("\nDetailed breakdown:")
    print(pd.crosstab(df['is_graspable'], df['has_collision']))

# Read the CSV
df = pd.read_csv("../training.csv")

# Print overall statistics
total = len(df)
training = df['is_used_in_training'].sum()
print(f"Dataset Overview:")
print(f"Total samples: {total:,}")
print(f"Training samples: {training:,} ({training/total*100:.2f}%)")
print(f"Non-training samples: {total-training:,} ({(total-training)/total*100:.2f}%)")

# Analyze both sets
analyze_grasp_results(df[df['is_used_in_training']], "Training Data")
analyze_grasp_results(df[~df['is_used_in_training']], "Non-Training Data")

In [ ]:
import pandas as pd
import torch
from core.visualize import check_collision

# Load the CSV
df = pd.read_csv('../training.csv')

# Filter for grasps that have collision but are graspable
interesting_grasps = df[
    (df['has_collision'] == True) & 
    (df['is_graspable'] == True) 
    # & 
    # (~df['was_skipped'])
]

print(f"Found {len(interesting_grasps)} grasps with collision that are graspable")

# Function to convert string representation of list back to tensor
def str_to_tensor(s):
    # Remove brackets and split by comma
    values = eval(s)  # Be careful with eval - use only on trusted data
    return torch.tensor(values)

# Visualize each interesting grasp
for idx, grasp in interesting_grasps.iterrows():
    print(f"\nVisualizing grasp {idx} for mesh: {grasp['mesh_path']}")
    
    # Convert stored data back to tensors
    rotation = str_to_tensor(grasp['grasp_rotation']).unsqueeze(0)  # Add batch dimension
    translation = str_to_tensor(grasp['grasp_translation']).unsqueeze(0)
    
    # Call check_collision to get the scene
    has_collision, scene, min_distance, is_graspable = check_collision(
        rotation,
        translation,
        "../"+grasp['mesh_path'],
        grasp['dataset_mesh_scale']
    )
    
    # Show the scene
    scene.show()
    
    break